# SOLA Method with Noise on a 3D Box Domain

## The Continuous Problem

We address the task of estimating a finite-dimensional **property** of an unknown spatial model from **noisy observations**. The model space is the $L^2$ space on a unit cube.

Let $m \in \mathcal{M} = L^2([0,1]^3)$ be the unknown model, $\tilde{\mathbf{d}} \in \mathcal{D} \cong \mathbb{R}^{N_d}$ be the observed data, and $\tilde{\mathbf{p}} \in \mathcal{P} \cong \mathbb{R}^{N_p}$ be the desired properties.

### Forward Operator

The forward map is

$$G : \mathcal{M} \to \mathcal{D}, \qquad [G(m)]_i = \int_{[0,1]^3} K_i(\mathbf{x})\, m(\mathbf{x})\, \mathrm{d}V$$

where $K_i$ are **Gaussian sensitivity kernels**. Noisy observations:

$$\tilde{\mathbf{d}} = G(\bar{m}) + \boldsymbol{\eta}, \qquad \boldsymbol{\eta} \sim \mathcal{N}(\mathbf{0}, \mathbf{C}_\mathcal{D})$$

### SOLA Approach

We do not reconstruct $\bar{m}$ directly. Instead we seek a surrogate operator $\mathbf{X}: \mathcal{D} \to \mathcal{P}$ such that $\mathcal{T} \approx \mathbf{X} G$, minimising

$$\min_{\mathbf{X}} \left[\|\mathcal{T} - \mathbf{X}G\|_{HS}^2 + \mathrm{Tr}(\mathbf{X}\mathbf{C}_\mathcal{D}\mathbf{X}^*)\right]$$

The **analytical solution** is

$$\boxed{\mathbf{X} = \mathcal{T}\, G^*\!\left(G G^* + \mathbf{C}_\mathcal{D}\right)^{-1}}$$

The property estimate is obtained by pushing the data measure forward:

$$\mu_\mathcal{P} = \mathbf{X}_* \mu_\mathcal{D} = \mathcal{N}(\mathbf{X}\tilde{\mathbf{d}},\; \mathbf{X}\mathbf{C}_\mathcal{D}\mathbf{X}^*)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

from pygeoinf import LinearOperator
from pygeoinf3d import BoxDomain, Lebesgue3D, Function3D
from pygeoinf3d.providers.function_providers import NormalModesProvider3D, GaussianProvider3D
from pygeoinf3d.solvers import DenseVolumeSolaSolver
from pygeoinf3d.plot import (
    plot_function3d_axis_slider,
    compare_function3d_axis_sliders,
    plot_function3d_axis_dropdown_slider,
    compare_panels_axis_dropdown_sliders
)

np.random.seed(42)

N_SLICES = 10     # slider resolution — keep low for fast rendering


## Spaces and Operators

We work on the unit cube $[0,1]^3$ with an $L^2$ model space.

* **$G$** — forward operator: $N_d$ Gaussian sensitivity kernels at random locations, built via `DenseVolumeSolaSolver`
* **$\mathcal{T}$** — target/property operator: $N_p$ tighter Gaussians on a $5\times5\times5$ regular grid


In [ ]:
# ── Domain & model space ─────────────────────────────────────────────────────
box   = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

QUAD_RESOLUTION = 20   # quadrature points per axis for DenseVolumeSolaSolver

# ── Forward operator G ────────────────────────────────────────────────────────
N_d = 1000
centers_G = np.random.uniform(0.1, 0.9, size=(N_d, 3))
sigmas_G  = np.random.uniform(0.05, 0.10, size=(N_d, 3))
fp_G     = NormalModesProvider3D(box, dim=N_d, freq_range=(1, 3), gaussian_width_percent_range=(5, 30))
solver_G = DenseVolumeSolaSolver(box, fp_G, resolution=QUAD_RESOLUTION)
G        = solver_G.forward_operator(space)   # LinearOperator: M → R^N_d

# ── Target/property operator T ────────────────────────────────────────────────
target_grid = np.array([[x, y, z]
                         for x in np.linspace(0.1, 0.9, 5)
                         for y in np.linspace(0.1, 0.9, 5)
                         for z in np.linspace(0.1, 0.9, 5)])
N_p = len(target_grid)   # 125  (5×5×5 grid)
fp_T     = GaussianProvider3D(box, centers=target_grid, sigmas=0.08, normalized=True)
solver_T = DenseVolumeSolaSolver(box, fp_T, resolution=QUAD_RESOLUTION)
T_op     = solver_T.forward_operator(space)   # LinearOperator: M → R^N_p

print(f"Forward operator G  : {space.__class__.__name__} -> R^{N_d}")
print(f"Target operator T   : {space.__class__.__name__} -> R^{N_p}")
print(f"Target locations    : {N_p} (5x5x5 grid)")


## Sensitivity and Target Kernel Visualisation

We visualise selected kernels as mid-plane ($z = 0.5$) slices.

In [ ]:
# ── All sensitivity kernels — dropdown + z-slider ────────────────────────────
g_kernels = [fp_G.get_function(i) for i in range(N_d)]
g_labels  = [f"G kernel {i}  ({centers_G[i,0]:.2f},{centers_G[i,1]:.2f},{centers_G[i,2]:.2f})"
             for i in range(N_d)]
_ = plot_function3d_axis_dropdown_slider(
    g_kernels[0:10], axis='z', n_slices=N_SLICES, resolution=80,
    labels=g_labels[0:10], title="Sensitivity kernels  (select from dropdown)",
    show_plot=True,
)

# ── All target kernels — dropdown + z-slider ──────────────────────────────────
t_kernels = [fp_T.get_function(k) for k in range(N_p)]
t_labels  = [f"T k={k} ({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})"
             for k in range(N_p)]
_ = plot_function3d_axis_dropdown_slider(
    t_kernels[0:10], axis='z', n_slices=N_SLICES, resolution=80,
    labels=t_labels[0:10], title="Target kernels  (select from dropdown)",
    show_plot=True,
)


## Synthetic Data

We define a multi-blob true model $m_{\text{true}}$ and generate noisy synthetic data
$$
d = G\,m_{\text{true}} + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, C_D).
$$

In [ ]:
def gaussian_blob(x, center, sigma):
    """Evaluate a normalised 3-D Gaussian at points x (shape n×3)."""
    diff = x - center[None, :]
    return np.exp(-0.5 * np.sum((diff / sigma) ** 2, axis=1))

def true_model_fn(x):
    """Two positive + one negative Gaussian blob."""
    if np.ndim(x) == 1:
        x = x[None, :]
    vals = (
        + 1.5 * gaussian_blob(x, np.array([0.30, 0.30, 0.50]), 0.12)
        + 1.0 * gaussian_blob(x, np.array([0.70, 0.65, 0.55]), 0.10)
        - 0.8 * gaussian_blob(x, np.array([0.55, 0.40, 0.45]), 0.08)
    )
    return vals.squeeze()

m_true = Function3D(box, evaluate_callable=true_model_fn)

# Clean and noisy data
d_bar     = G(m_true)
noise_std = 0.05 * np.abs(d_bar).max()
d_tilde   = d_bar + np.random.normal(0.0, noise_std, size=N_d)

print(f"Noise std : {noise_std:.4f}")
print(f"SNR       : {np.linalg.norm(d_bar) / np.linalg.norm(d_tilde - d_bar):.1f}")

# ── Interactive z-slider for true model ───────────────────────────────────────
plot_function3d_axis_slider(
    m_true, axis='z', n_slices=N_SLICES, resolution=80,
    title="True model  (z-axis slider)", show_plot=False,
)

## Data Noise Model

We model the data noise as i.i.d. Gaussian:
$$
\tilde{\mathbf{d}} = G(\bar{m}) + \boldsymbol{\eta}, \qquad \boldsymbol{\eta} \sim \mathcal{N}(\mathbf{0},\, \mathbf{C}_D), \qquad \mathbf{C}_D = \sigma_n^2 \mathbf{I}.
$$
The covariance matrix is passed directly to the solver — no separate `GaussianMeasure` object is needed.


In [ ]:
C_D_matrix = noise_std**2 * np.eye(N_d)
# Replace with any SPD matrix for correlated noise.
print(f"Data noise std      : {noise_std:.2e}")
print(f"Data noise variance : {C_D_matrix[0, 0]:.2e}")


## SOLA Operator Construction

The SOLA solution operator is
$$
\mathbf{X} = \mathcal{T}\, G^*\!\left(G G^* + \mathbf{C}_D\right)^{-1}.
$$

`DenseVolumeSolaSolver.compute_sola_matrix(solver_T, noise_cov)` builds this
as a dense $(N_p \times N_d)$ matrix using the push-through identity so the
solve is always in the small quadrature space $\mathbb{R}^{N_q}$ regardless of $N_d$.


In [ ]:
X_mat = solver_G.compute_sola_matrix(solver_T, noise_cov=C_D_matrix)  # (N_p, N_d)
print("SOLA matrix X assembled.")
print(f"  shape    : {X_mat.shape}  (N_p x N_d)")
print(f"  domain   : R^{N_d}  (data space)")
print(f"  codomain : R^{N_p}  (property space)")


## Model Reconstruction (Optional)

The generalised inverse $G^{-1}$ also provides an approximate model reconstruction
$\hat{m} = G^{-1}(\tilde{d})$.
This is a Function3D living in the model space.

In [ ]:
m_hat    = solver_G.reconstruct(d_tilde, noise_cov=C_D_matrix)   # Function3D

# Data residual
residual = np.linalg.norm(G(m_hat) - d_tilde) / np.linalg.norm(d_tilde)
print(f"Relative data residual: {residual:.4f}")

# ── Side-by-side interactive slider: true model vs. reconstruction ───────────
compare_function3d_axis_sliders(
    [m_true, m_hat], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["True model", "Reconstruction"],
    show_plot=False,
    width=1000, height=500,
)


## Property Inference with Uncertainty

The SOLA estimator $\hat{p} = X(\tilde{d})$ is a linear function of the data.
Because the data follows a Gaussian measure $\mu_D$, the inferred properties also follow a Gaussian:
$$
\mu_P = X_* \mu_D = \mathcal{N}(X\tilde{d},\;\; X C_D X^*).
$$

The posterior standard deviations quantify the propagated data noise.

In [ ]:
p_mean    = X_mat @ d_tilde                        # (N_p,)
P_cov_mat = X_mat @ C_D_matrix @ X_mat.T           # X C_D X^T  (N_p x N_p)
p_std     = np.sqrt(np.diag(P_cov_mat))
p_true    = T_op(m_true)                           # (N_p,)

N_PRINT = min(20, N_p)
print(f"(showing first {N_PRINT} of {N_p} properties)")
print("Property  |  true    |  mean    |  std")
print("-" * 48)
for k in range(N_PRINT):
    cx, cy, cz = target_grid[k]
    print(f"  k={k}  ({cx:.2f},{cy:.2f},{cz:.2f})  |  {p_true[k]:.4f}  |  {p_mean[k]:.4f}  |  {p_std[k]:.4f}")

# ── Helper: wrap N_p property values as a Function3D on the target grid ──────
_grid_ax = np.unique(target_grid[:, 0])
_n_ax    = len(_grid_ax)

def _make_prop_fn(values):
    grid_vals = values.reshape(_n_ax, _n_ax, _n_ax)
    _lo, _hi  = _grid_ax[0], _grid_ax[-1]
    _interp   = RegularGridInterpolator(
        (_grid_ax, _grid_ax, _grid_ax), grid_vals,
        method='linear', bounds_error=False, fill_value=0.0,
    )
    def _fn(x):
        pts_c = np.clip(np.atleast_2d(x), _lo, _hi)
        return _interp(pts_c).squeeze()
    return Function3D(box, evaluate_callable=_fn)

p_mean_field = _make_prop_fn(p_mean)
p_2std_field = _make_prop_fn(2 * p_std)

# ── 3D slider: SOLA estimate and uncertainty side-by-side ────────────────────
compare_function3d_axis_sliders(
    [p_mean_field, p_2std_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["SOLA estimate (mean)", "Uncertainty (2σ)"],
    show_plot=False,
    width=1000, height=500,
)


## Unimodularity Constraint

A SOLA estimator is **unimodular** if $XG \mathbf{1} = \mathbf{1}$, i.e., a uniform model maps to unit
property values.  The unconstrained $X$ generally violates this.  We apply a rank-1 correction:
$$
X_{\text{uni}} = X - \frac{(X\,G\mathbf{1} - \mathbf{1})\,u^T}{\langle G\mathbf{1},\, u\rangle}, \quad
u = W^{-1}(G\mathbf{1}).
$$

In [ ]:
ones = Function3D(box, evaluate_callable=lambda x: np.ones(np.atleast_2d(x).shape[0]))

v    = G(ones)                                          # G applied to constant field -> R^{N_d}
w    = np.ones(N_p)                                     # desired unimodular output
u    = solver_G.solve(v, noise_cov=C_D_matrix)          # (C_D + GG*)^{-1} v -> R^{N_d}
beta = float(np.dot(v, u))                              # <v, u> scalar

# Rank-1 correction  (N_p x N_d)
correction_matrix = np.outer((X_mat @ v - w) / beta, u)
X_mat_uni = X_mat - correction_matrix

# Verify unimodularity
check = X_mat_uni @ v
print(f"X_uni(G 1) — should equal ones (first 5): {np.asarray(check).round(4)[:5]}")

# Property estimates and uncertainties with unimodular X
p_mean_uni = X_mat_uni @ d_tilde
P_cov_uni  = X_mat_uni @ C_D_matrix @ X_mat_uni.T
p_std_uni  = np.sqrt(np.diag(P_cov_uni))

p_mean_uni_field = _make_prop_fn(p_mean_uni)
p_2std_uni_field = _make_prop_fn(2 * p_std_uni)

# ── Mean: unconstrained vs unimodular ─────────────────────────────────────────
compare_function3d_axis_sliders(
    [p_mean_field, p_mean_uni_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["Mean — unconstrained X", "Mean — unimodular X"],
    show_plot=False,
    width=1000, height=500,
)

# ── Uncertainty: unconstrained vs unimodular ──────────────────────────────────
compare_function3d_axis_sliders(
    [p_2std_field, p_2std_uni_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["2σ — unconstrained X", "2σ — unimodular X"],
    show_plot=False,
    width=1000, height=500,
)

# ── Unimodular estimate + uncertainty ─────────────────────────────────────────
compare_function3d_axis_sliders(
    [p_mean_uni_field, p_2std_uni_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["SOLA estimate (mean) — unimodular X", "Uncertainty (2σ) — unimodular X"],
    show_plot=False,
    width=1000, height=500,
)


## Resolving Kernels

The **resolving kernel** for property $k$ is the function
$$
A_k(x) = \bigl(X G\bigr)^*(e_k)(x) = \bigl(G^*\,X^*\,e_k\bigr)(x),
$$
where $e_k$ is the $k$-th standard basis vector in property space $P$.
Each $A_k$ is a weighted superposition of the sensitivity kernels; ideally it closely
approximates the corresponding target kernel $T_k$.

In [ ]:
# (X G)^* = G^* X^*   maps  P -> M
X_op      = LinearOperator.from_matrix(G.codomain, T_op.codomain, X_mat)
A_adjoint = (X_op @ G).adjoint

# Small preview: 5 points along the main diagonal of the 5x5x5 grid
#   k = ix*25 + iy*5 + iz  for  ix=iy=iz = 0,1,2,3,4
PREVIEW_IDX = [0, 31, 62, 93, 124]   # (0.1,0.1,0.1) ... (0.9,0.9,0.9)

def _e(n, k):
    """Cheap basis vector - avoids allocating an nxn identity matrix."""
    v = np.zeros(n)
    v[k] = 1.0
    return v

rk_fns  = [A_adjoint(_e(N_p, k)) for k in PREVIEW_IDX]
tgt_fns = [fp_T.get_function(k)  for k in PREVIEW_IDX]

rk_labels  = [
    f"Resolving k={k} "
    f"({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})"
    for k in PREVIEW_IDX
]
tgt_labels = [
    f"Target k={k} "
    f"({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})"
    for k in PREVIEW_IDX
]

compare_panels_axis_dropdown_sliders(
    panel_functions=[rk_fns, tgt_fns],
    panel_labels=[rk_labels, tgt_labels],
    panel_titles=["Resolving kernels", "Target kernels"],
    axis="z",
    n_slices=N_SLICES,
    resolution=80,
    show_plot=True,
)

print("Each resolving kernel should be concentrated near its matching target centre.")
